# Computational repeatability and requirement checks

**Purpose:** Repeat the foundational procedure, verify numerical agreement and reconcile the evidence for nineteen requirements.

**Required inputs:** Prepared Baček HDF5; applicable results ZIPs from 02–14.

**Outputs:** Repeatability, metric-recalculation and requirements evidence.

**Usage:** Run after the upstream results have been generated. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Environment, source and verification settings
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, io, json, os, platform, sys, time, zipfile
import h5py
import numpy as np
import scipy
import sklearn
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
OUT = Path('15_Bacek_Computational_Repeatability_Requirements_Audit_results')
ARCHIVE = Path(str(OUT) + '.zip')
AUTHORITATIVE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
CHANNELS = ('hip_left','hip_right','knee_left','knee_right','ankle_left','ankle_right',
            'pelvis_sagittal','pelvis_frontal','pelvis_transverse')
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
PCA_COUNTS = (5,10,20,40)
SPEEDS = (1.1,0.8,0.4)
CADENCES = (0.90,0.95,1.00,1.10,1.20)
OUTER_FOLDS = {
    1: ('S08','S10','S16','S21'), 2: ('S11','S13','S17','S20'),
    3: ('S01','S02','S05','S18'), 4: ('S03','S07','S12','S14','S19'),
    5: ('S04','S06','S09','S15')}
FOLD_OF = {p: f for f, people in OUTER_FOLDS.items() for p in people}
EXPECTED_COUNTS = (55,53,56,65,55)
PROCEDURES = ('B0','B1','B2','W-ALL9_FIXED','D-ALL9_FIXED','P-ALL9_FIXED','G_SELECTED')

# Prespecified pilot-to-confirmation numerical rule (not a physiological accuracy threshold).
# The strict floor is intentionally ~10^3-10^4 times float64 machine epsilon at W/kg scale;
# an observed pilot difference is given a 10-fold headroom. A hard cap prevents an
# unexpectedly large disagreement from automatically becoming an acceptable tolerance.
PRECISION_FLOOR_W_PER_KG = 1e-12
PILOT_HEADROOM_FACTOR = 10.0
MAX_PERMITTED_PROPOSAL_W_PER_KG = 1e-8

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def require(cond, message):
    if not bool(cond):
        raise AssertionError(message)

def read_zip_csv(z, name):
    return list(csv.DictReader(io.StringIO(z.read(name).decode('utf-8-sig'))))

def float_or_none(v):
    return None if v in ('',None) else float(v)

def rmse(y, p):
    e = np.asarray(p,dtype=np.float64)-np.asarray(y,dtype=np.float64)
    return float(np.sqrt(np.mean(e**2,dtype=np.float64)))

def performance(y,p):
    y,p=np.asarray(y,dtype=np.float64),np.asarray(p,dtype=np.float64)
    e=p-y;den=np.sum((y-y.mean())**2,dtype=np.float64)
    return {'n_trials':len(y),'RMSE_W_per_kg':rmse(y,p),
            'MAE_W_per_kg':float(np.mean(np.abs(e),dtype=np.float64)),
            'bias_W_per_kg':float(np.mean(e,dtype=np.float64)),
            'R2':float(1-np.sum(e*e,dtype=np.float64)/den) if den>0 else None}

def write_csv(path,records,fields):
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        w.writeheader();w.writerows(records)

PACKAGE_VERSIONS={'python':platform.python_version(),'numpy':np.__version__,
    'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,
    'h5py':h5py.__version__,'os':platform.platform(),
    'cpu_count':os.cpu_count(),'kernel_python_executable':sys.executable}
try:
    from threadpoolctl import threadpool_info
    THREADPOOLS=threadpool_info()
except ImportError:
    THREADPOOLS=[]
print('Notebook 15 —',PACKAGE_VERSIONS)

## Input evidence inventory


In [ ]:
# 2 — Read the canonical Notebook 04 evidence and inventory all earlier archives
ARTIFACTS = {
  2:'02_Bacek_Numerical_Integrity_Preflight_results.zip',
  3:'03_Bacek_Feature_Pipeline_Verification_results.zip',
  4:'04_Bacek_Foundational_Predictive_Models_results.zip',
  5:'05_Bacek_Incremental_Gait_Information_results.zip',
  6:'06_Bacek_Biological_Information_Comparison_results.zip',
  7:'07_Bacek_Anatomical_Gait_Information_results.zip',
  8:'08_Bacek_Descriptor_Family_Comparison_results.zip',
  9:'09_Bacek_Ridge_vs_RBF_SVR_Comparison_results.zip',
 10:'10_Bacek_Regularisation_Training_Participant_Sufficiency_results.zip',
 11:'11_Bacek_Known_Person_Calibration_results.zip',
 12:'12_Bacek_285_Trial_Eligibility_Sensitivity_results.zip',
 13:'13_Bacek_Participant_Condition_Error_Diagnostics_results.zip',
 14:'14_Bacek_Clustered_Uncertainty_Influence_Selection_Stability_results.zip'}
archives={}; summaries={}; inventory=[]
for n, filename in ARTIFACTS.items():
    path=Path(filename)
    if not path.is_file():
        inventory.append({'notebook':n,'filename':filename,'sha256':'',
                          'n_files':0,'reported_status':'MISSING','verification_rows':0,
                          'failed_checks':0,'source_sha256_match':''})
        continue
    digest=sha256_file(path)
    with zipfile.ZipFile(path) as z:
        bad=z.testzip()
        require(bad is None,f'ZIP {filename}: corrupted entry {bad}')
        summ_name=next((a for a in z.namelist() if a.endswith(('run_summary.json','verification_summary.json','preflight_summary.json'))),None)
        summ=json.loads(z.read(summ_name)) if summ_name else {}
        summary_status=summ.get('status',summ.get('verification_status','NOT_REPORTED'))
        checkname=next((a for a in z.namelist() if a.endswith(('verification_checks.csv','synthetic_verification_checks.csv'))),None)
        checks=read_zip_csv(z,checkname) if checkname else []
        n_failed=sum(r.get('status','PASS')!='PASS' for r in checks)
        source=summ.get('source_sha256','')
        src_match=(source==AUTHORITATIVE_SHA256) if source else None
        inventory.append({'notebook':n,'filename':filename,'sha256':digest,
                          'n_files':len(z.namelist()),'reported_status':summary_status,
                          'verification_rows':len(checks),'failed_checks':n_failed,
                          'source_sha256_match':src_match})
        summaries[n]=summ
    archives[n]=path

require(4 in archives,'Canonical Notebook 04 ZIP is required')
require(summaries[4].get('status')=='PASS','Notebook 04 must report PASS')
require(inventory[2]['reported_status']=='PASS' if 2 in archives else True,
        'Notebook 02 preflight must report PASS when present')
with zipfile.ZipFile(archives[4]) as z:
    reference_ledger=read_zip_csv(z,'held_out_predictions.csv')
    reference_selected=read_zip_csv(z,'selected_models.csv')
    reference_inner=read_zip_csv(z,'inner_fold_manifest.csv')
    reference_outer=read_zip_csv(z,'outer_fold_manifest.csv')
    reference_trials=read_zip_csv(z,'eligible_trial_ledger.csv')
    reference_pooled=read_zip_csv(z,'pooled_metrics.csv')
    reference_scores=read_zip_csv(z,'pooled_candidate_scores.csv')

require(len(reference_ledger)==284*7,'Reference predictions incomplete')
require(len(reference_selected)==30 and len(reference_inner)==20 and len(reference_scores)==240,
        'Reference model selection manifests incomplete')
ref_pred={(r['procedure'],r['trial_uid']):r for r in reference_ledger}
require(len(ref_pred)==len(reference_ledger),'Duplicate reference prediction keys')
ref_sel={(int(r['outer_fold']),r['procedure']):r for r in reference_selected}
ref_score={(int(r['outer_fold']),r['representation'],float(r['alpha']),
            float_or_none(r['n_components'])):r for r in reference_scores}
ref_trial={r['trial_uid']:r for r in reference_trials}
require(len(ref_trial)==284,'Reference trial ledger must be unique')
ref_env=summaries[4]['software']
env_matches={k:PACKAGE_VERSIONS.get(k)==ref_env.get(k)
             for k in ('python','numpy','scipy','scikit_learn','h5py','os','kernel_python_executable')}
print('Evidence ZIPs present:',len(archives),'/ 13')
print('Same Notebook 04 environment:',all(env_matches.values()),env_matches)
print('Reference:',len(reference_ledger),'predictions,',len(reference_selected),'selections')

## Repeated foundational modelling


In [ ]:
# 3 — Source reading and exact trial-level feature definitions

def features_from_trial(trial):
    means=[]; n_cycles=None
    for channel in CHANNELS:
        require(channel in trial,f'Missing channel {channel}')
        angle=np.asarray(trial[channel][()],dtype=np.float64)
        require(angle.ndim==2 and angle.shape[1]==101,'Invalid angle shape')
        require(np.isfinite(angle).all(),'Nonfinite gait angles')
        if n_cycles is None: n_cycles=angle.shape[0]
        require(angle.shape[0]==n_cycles,'Inconsistent cycle count')
        means.append(angle.mean(axis=0,dtype=np.float64))
    m=np.stack(means); w=m.reshape(909)
    d=[]
    for channel in m:
        for q in range(4):
            d.append(float(np.trapezoid(channel[25*q:25*q+26],dx=1.0)/25.0))
        d.append(float(np.std(channel,ddof=0)))
        d.append(float(np.max(channel)-np.min(channel)))
    return w,np.asarray(d,dtype=np.float64),n_cycles

def read_primary_features():
    require(H5.is_file(),f'Corrected HDF5 not found: {H5}')
    source_hash=sha256_file(H5)
    require(source_hash==AUTHORITATIVE_SHA256,'Authoritative HDF5 SHA-256 mismatch')
    rows,w_rows,d_rows=[],[],[]
    with h5py.File(H5,'r') as h5:
        subjects=sorted(h5['subjects'])
        require(subjects==sorted(FOLD_OF),'Participant IDs differ from frozen folds')
        for subject in subjects:
            for trial_id in sorted(h5['subjects'][subject]['trials']):
                if not (trial_id.startswith('T') and trial_id[1:].isdigit()
                        and 1<=int(trial_id[1:])<=15):
                    continue
                trial=h5['subjects'][subject]['trials'][trial_id]
                uid=f'{subject}_{trial_id}'
                w,d,n=features_from_trial(trial)
                require(n>=20,f'{uid}: fewer than 20 prepared cycles')
                for attr in ('valid','comparison_eligible','unrestricted_comparison_eligible'):
                    require(attr in trial.attrs and
                            np.asarray(trial.attrs[attr]).reshape(-1)[0]==1,
                            f'{uid}: invalid {attr}')
                require('metcost_mean' in trial, f'{uid}: missing corrected target')
                target=np.asarray(trial['metcost_mean'][()])
                require(target.size==1 and np.issubdtype(target.dtype,np.number)
                        and np.isrealobj(target),f'{uid}: target format mismatch')
                yy=float(target.reshape(-1)[0])
                require(np.isfinite(yy) and yy>0, f'{uid}: invalid target')
                require('original_cycle_number' in trial,f'{uid}: missing cycle provenance')
                indices=np.asarray(trial['original_cycle_number'][()]).reshape(-1)
                require(len(indices)==n and np.all(np.isfinite(indices))
                        and np.all(indices==np.floor(indices))
                        and np.all(np.diff(indices)>0),f'{uid}: invalid provenance')
                if uid=='S06_T11':
                    require(n==85,'S06_T11 must contain 85 retained cycles')
                    continue
                require(n>=100,f'{uid}: primary cycle criterion violated')
                block,cad_idx=divmod(int(trial_id[1:])-1,5)
                rows.append({'trial_uid':uid,'subject_id':subject,'trial_id':trial_id,
                             'outer_fold':FOLD_OF[subject],'retained_cycles':n,
                             'speed_m_s':SPEEDS[block],'relative_cadence':CADENCES[cad_idx],
                             'condition_index':int(trial_id[1:])-1,'speed_index':block,
                             'observed_W_per_kg':yy})
                w_rows.append(w);d_rows.append(d)
    require(len(rows)==284 and len({r['trial_uid'] for r in rows})==284,'Invalid primary ledger')
    require(sum(r['retained_cycles'] for r in rows)==60757,'Incorrect cycle total')
    require([sum(r['outer_fold']==i for r in rows) for i in OUTER_FOLDS]==list(EXPECTED_COUNTS),
            'Incorrect outer-fold trial counts')
    require(len({r['subject_id'] for r in rows})==21,'Incorrect participant count')
    for r in rows:
        prior=ref_trial.get(r['trial_uid'])
        require(prior is not None and prior['subject_id']==r['subject_id']
                and int(prior['outer_fold'])==r['outer_fold']
                and int(prior['retained_cycles'])==r['retained_cycles']
                and float(prior['observed_W_per_kg'])==r['observed_W_per_kg'],
                f'{r["trial_uid"]}: trial not identical to Notebook 04')
    Xw=np.stack(w_rows); Xd=np.stack(d_rows)
    require(Xw.shape==(284,909) and Xd.shape==(284,54),'Unexpected feature dimensions')
    Xb1=np.eye(3,dtype=np.float64)[[r['speed_index'] for r in rows]]
    Xb2=np.eye(15,dtype=np.float64)[[r['condition_index'] for r in rows]]
    return {'rows':rows,'Xw':Xw,'Xd':Xd,'Xb1':Xb1,'Xb2':Xb2,
            'groups':np.array([r['subject_id'] for r in rows]),
            'uids':np.array([r['trial_uid'] for r in rows]),
            'y':np.array([r['observed_W_per_kg'] for r in rows],dtype=np.float64),
            'source_sha256':source_hash}

def candidates():
    baselines=[{'representation':name,'alpha':alpha,'n_components':None,'n_features':nf}
               for name,nf in [('B1',3),('B2',15)] for alpha in ALPHAS]
    gait=([{'representation':name,'alpha':alpha,'n_components':None,'n_features':nf}
            for name,nf in [('W-ALL9',909),('D-ALL9',54)] for alpha in ALPHAS]
           + [{'representation':'P-ALL9','alpha':alpha,'n_components':k,'n_features':k}
              for k in PCA_COUNTS for alpha in ALPHAS])
    return baselines,gait

def model_input(candidate,data):
    return {'B1':data['Xb1'],'B2':data['Xb2'],'W-ALL9':data['Xw'],
            'D-ALL9':data['Xd'],'P-ALL9':data['Xw']}[candidate['representation']]

def model_pipeline(candidate):
    rep=candidate['representation']
    if rep in ('B1','B2'):
        return Ridge(alpha=candidate['alpha'],fit_intercept=True)
    steps=[('scale',StandardScaler())]
    if rep=='P-ALL9':
        steps.append(('pca',PCA(n_components=candidate['n_components'],
                                 whiten=False,random_state=2026)))
    steps.append(('ridge',Ridge(alpha=candidate['alpha'],fit_intercept=True)))
    return Pipeline(steps)

def choose(scores,gait=False):
    if gait:
        return min(scores,key=lambda r:(r['inner_pooled_RMSE_W_per_kg'],r['n_features'],
            r['representation'],r['n_components'] or 0,r['alpha']))
    return min(scores,key=lambda r:(r['inner_pooled_RMSE_W_per_kg'],r['alpha']))

base_candidates,gait_candidates=candidates()
require(len(base_candidates)==12 and len(gait_candidates)==36,'Candidate grid changed')

In [ ]:
# 4 — Full end-to-end repeat: source read, feature construction, all inner fits, refits, predictions

def run_complete_repeat(run_id):
    started_at=datetime.now(timezone.utc).isoformat()
    start=time.perf_counter()
    data=read_primary_features()
    y, groups, uids = data['y'],data['groups'],data['uids']
    results={'run_id':run_id,'predictions':[],'selected':[],'scores':[],
             'inner_folds':[],'outer_folds':[],'features':data,
             'started_at_utc':started_at}
    for outer_fold,test_participants in OUTER_FOLDS.items():
        ti=np.flatnonzero(np.isin(groups,test_participants))
        di=np.flatnonzero(~np.isin(groups,test_participants))
        require(len(ti)==EXPECTED_COUNTS[outer_fold-1], 'Outer fold size mismatch')
        require(set(groups[ti]).isdisjoint(groups[di]),'Outer participant overlap')
        inner=[]; covered=set()
        for inner_fold,(trloc,valoc) in enumerate(
                GroupKFold(n_splits=4).split(data['Xw'][di],y[di],groups[di]),1):
            tr,va=di[trloc],di[valoc]
            train_people=set(groups[tr]);val_people=set(groups[va])
            require(train_people.isdisjoint(val_people),'Inner participant overlap')
            require(covered.isdisjoint(va.tolist()),'Duplicate validation trial')
            covered.update(va.tolist());inner.append((tr,va))
            man={'outer_fold':outer_fold,'inner_fold':inner_fold,
                 'train_participants':';'.join(sorted(train_people)),
                 'validation_participants':';'.join(sorted(val_people)),
                 'outer_test_participants':';'.join(sorted(test_participants)),
                 'n_train_trials':len(tr),'n_validation_trials':len(va)}
            ref=next(r for r in reference_inner if int(r['outer_fold'])==outer_fold
                     and int(r['inner_fold'])==inner_fold)
            for name in ('train_participants','validation_participants','outer_test_participants'):
                require(man[name]==ref[name],f'Inner fold changed: {outer_fold}/{inner_fold}/{name}')
            results['inner_folds'].append(man)
        require(covered==set(di.tolist()),'Incomplete inner folds')
        results['outer_folds'].append({'outer_fold':outer_fold,'n_dev_trials':len(di),
             'n_test_trials':len(ti),'test_participants':';'.join(sorted(test_participants))})
        scored=[]; pos={int(ix):k for k,ix in enumerate(di)}
        for candidate in base_candidates+gait_candidates:
            X=model_input(candidate,data)
            preds=np.full(len(di),np.nan,dtype=np.float64)
            for tr,va in inner:
                fitted=model_pipeline(candidate)
                fitted.fit(X[tr],y[tr]);v=np.asarray(fitted.predict(X[va]),dtype=np.float64)
                require(np.isfinite(v).all(),'Nonfinite inner predictions')
                for ix,pr in zip(va,v): preds[pos[int(ix)]]=pr
            require(np.isfinite(preds).all(),'Incomplete inner candidate predictions')
            score={'outer_fold':outer_fold,**candidate,
                   'inner_pooled_RMSE_W_per_kg':rmse(y[di],preds)}
            scored.append(score);results['scores'].append(score)
        byrep={rep:[s for s in scored if s['representation']==rep]
               for rep in ('B1','B2','W-ALL9','D-ALL9','P-ALL9')}
        chosen={'B1':choose(byrep['B1']),'B2':choose(byrep['B2']),
                'W-ALL9_FIXED':choose(byrep['W-ALL9'],gait=True),
                'D-ALL9_FIXED':choose(byrep['D-ALL9'],gait=True),
                'P-ALL9_FIXED':choose(byrep['P-ALL9'],gait=True),
                'G_SELECTED':choose([s for s in scored if s['representation'] in
                     ('W-ALL9','D-ALL9','P-ALL9')],gait=True)}
        predictions={'B0':np.full(len(ti),float(np.mean(y[di],dtype=np.float64)))}
        cache={}
        for procedure,candidate in chosen.items():
            key=(candidate['representation'],candidate['alpha'],candidate['n_components'])
            if key not in cache:
                X=model_input(candidate,data);fit=model_pipeline(candidate)
                fit.fit(X[di],y[di]);cache[key]=np.asarray(fit.predict(X[ti]),dtype=np.float64)
            predictions[procedure]=cache[key]
            results['selected'].append({'outer_fold':outer_fold,'procedure':procedure,
                                       **candidate,'inner_pooled_RMSE_W_per_kg':
                                       candidate['inner_pooled_RMSE_W_per_kg']})
        for procedure in PROCEDURES:
            require(np.isfinite(predictions[procedure]).all(),'Nonfinite held-out prediction')
            for j,ix in enumerate(ti):
                results['predictions'].append({'run_id':run_id,'procedure':procedure,
                    'outer_fold':outer_fold,'subject_id':str(groups[ix]),
                    'trial_uid':str(uids[ix]),'observed_W_per_kg':float(y[ix]),
                    'predicted_W_per_kg':float(predictions[procedure][j])})
        print(f'{run_id}: outer fold {outer_fold} — '
              f'G={chosen["G_SELECTED"]["representation"]}, '
              f'{len(ti)} held-out trials')
    require(len(results['predictions'])==1988 and len(results['selected'])==30
            and len(results['scores'])==240,'Incomplete complete-run evidence')
    results['elapsed_seconds']=time.perf_counter()-start
    results['finished_at_utc']=datetime.now(timezone.utc).isoformat()
    return results

def compare_run_to_reference(run):
    result=[];keys=set()
    for row in run['predictions']:
        key=(row['procedure'],row['trial_uid'])
        keys.add(key);ref=ref_pred[key]
        require(int(ref['outer_fold'])==row['outer_fold'] and
                ref['subject_id']==row['subject_id'] and
                float(ref['observed_W_per_kg'])==row['observed_W_per_kg'],
                'Reference evaluation keys or targets changed')
        original=float(ref['predicted_W_per_kg'])
        result.append({'run_id':run['run_id'],'procedure':row['procedure'],
            'trial_uid':row['trial_uid'],'outer_fold':row['outer_fold'],
            'reference_prediction_W_per_kg':original,
            'repeated_prediction_W_per_kg':row['predicted_W_per_kg'],
            'absolute_difference_W_per_kg':abs(original-row['predicted_W_per_kg'])})
    require(keys==set(ref_pred),'Repeat prediction keys differ from reference')
    selection=[]
    for row in run['selected']:
        ref=ref_sel[(row['outer_fold'],row['procedure'])]
        same=(row['representation']==ref['representation'] and
              row['alpha']==float(ref['alpha']) and
              row['n_components']==float_or_none(ref['n_components']))
        selection.append({'run_id':run['run_id'],'outer_fold':row['outer_fold'],
            'procedure':row['procedure'],'matches_reference':same,
            'reference_representation':ref['representation'],
            'repeated_representation':row['representation'],
            'reference_alpha':ref['alpha'],'repeated_alpha':row['alpha'],
            'reference_n_components':ref['n_components'],
            'repeated_n_components':row['n_components']})
    score_differences=[]
    for row in run['scores']:
        key=(row['outer_fold'],row['representation'],row['alpha'],row['n_components'])
        ref=ref_score[key]
        score_differences.append({'run_id':run['run_id'],
             'outer_fold':row['outer_fold'],'representation':row['representation'],
             'alpha':row['alpha'],'n_components':row['n_components'],
             'absolute_difference_W_per_kg':abs(row['inner_pooled_RMSE_W_per_kg']-
                                          float(ref['inner_pooled_RMSE_W_per_kg']))})
    return {'prediction_diffs':result,'selections':selection,'score_diffs':score_differences,
            'max_prediction_abs_difference_W_per_kg':max(x['absolute_difference_W_per_kg'] for x in result),
            'max_score_abs_difference_W_per_kg':max(x['absolute_difference_W_per_kg'] for x in score_differences),
            'all_selected_configs_match':all(r['matches_reference'] for r in selection)}

## Repeatability pilot


In [ ]:
# 5 — Pilot full repeat, numerical calibration and pre-confirmation record
pilot=run_complete_repeat('PILOT')
pilot_comp=compare_run_to_reference(pilot)
pilot_max=pilot_comp['max_prediction_abs_difference_W_per_kg']
proposed_tolerance=max(PRECISION_FLOOR_W_PER_KG,PILOT_HEADROOM_FACTOR*pilot_max)
tolerance_proposal_valid=(proposed_tolerance<=MAX_PERMITTED_PROPOSAL_W_PER_KG
                          and pilot_comp['all_selected_configs_match']
                          and all(env_matches.values()))
if not tolerance_proposal_valid:
    proposed_tolerance=None
pilot_record={'pilot_finished_at_utc':pilot['finished_at_utc'],
              'pilot_max_prediction_difference_W_per_kg':pilot_max,
              'pilot_max_inner_score_difference_W_per_kg':pilot_comp['max_score_abs_difference_W_per_kg'],
              'pilot_all_model_selections_match':pilot_comp['all_selected_configs_match'],
              'precision_floor_W_per_kg':PRECISION_FLOOR_W_PER_KG,
              'headroom_factor':PILOT_HEADROOM_FACTOR,
              'protective_ceiling_W_per_kg':MAX_PERMITTED_PROPOSAL_W_PER_KG,
              'rc3_tolerance_proposal_W_per_kg':proposed_tolerance,
              'status':'PROPOSED_FOR_REPORT_REVIEW' if tolerance_proposal_valid else 'REVIEW_REQUIRED'}
print('Pilot max absolute difference:',pilot_max,'W/kg')
print('Proposed RC-3 tolerance:',proposed_tolerance,'W/kg')
print('Pilot status:',pilot_record['status'])

## Repeatability confirmation


In [ ]:
# 6 — Confirmatory full repeat with the tolerance already recorded in pilot_record
confirm=run_complete_repeat('CONFIRM')
confirm_comp=compare_run_to_reference(confirm)
confirm_diffs={ (r['procedure'],r['trial_uid']):r['repeated_prediction_W_per_kg']
                for r in confirm_comp['prediction_diffs'] }
pilot_diffs={ (r['procedure'],r['trial_uid']):r['repeated_prediction_W_per_kg']
              for r in pilot_comp['prediction_diffs'] }
pilot_vs_confirm=max(abs(pilot_diffs[k]-confirm_diffs[k]) for k in pilot_diffs)
if proposed_tolerance is None:
    confirm_status='REVIEW_REQUIRED_NO_VALID_TOLERANCE'
else:
    confirm_status=('CONFIRMATORY_CHECK_PASS' if (
        confirm_comp['all_selected_configs_match']
        and confirm_comp['max_prediction_abs_difference_W_per_kg']<=proposed_tolerance
        and pilot_vs_confirm<=proposed_tolerance)
        else 'CONFIRMATORY_CHECK_FAIL')
print('Confirmatory max vs canonical:',confirm_comp['max_prediction_abs_difference_W_per_kg'])
print('Confirmatory max vs pilot:',pilot_vs_confirm)
print('Repeatability confirmation:',confirm_status)
print('Formal RC-3 requirement status: PENDING — record justified tolerance in Chapter 3')

## Independent metric reconciliation


In [ ]:
# 7 — Independently recompute the main saved prediction metrics for archived experiments
metric_audit=[];issues=[]
for n in (4,5,6,7,8,9,10,11,12):
    if n not in archives:
        issues.append(f'Notebook {n:02d}: missing prediction evidence archive')
        continue
    with zipfile.ZipFile(archives[n]) as z:
        if n==10:
            pred_name='learning_curve_predictions.csv';metric_name='learning_curve_metrics.csv'
            group_keys=['budget_group','subset_replicate']
        elif n==11:
            pred_name='held_out_predictions.csv';metric_name='pooled_metrics.csv'
            # Personal report pooled metrics combine budgets under the identifier
            group_keys=['procedure','K']
        else:
            pred_name='held_out_predictions.csv';metric_name='pooled_metrics.csv'
            group_keys=['group_id' if n==7 else 'family_id' if n==8 else 'procedure']
        preds=read_zip_csv(z,pred_name);metrics=read_zip_csv(z,metric_name)
    # E1.3 already identifies the K-specific procedure in the procedure field.
    if n==11:
        group_keys=['procedure']
    grouped={}
    for row in preds:
        key=tuple(row.get(k,'') for k in group_keys)
        grouped.setdefault(key,[]).append(row)
    saved={tuple(m.get(k,'') for k in group_keys):m for m in metrics}
    for key,rs in grouped.items():
        actual=np.asarray([float(r['observed_W_per_kg']) for r in rs])
        predicted=np.asarray([float(r['predicted_W_per_kg']) for r in rs])
        measured=performance(actual,predicted)
        reference=saved.get(key)
        ok=reference is not None
        if ok:
            for name,val in measured.items():
                if val is None: continue
                saved_field={'RMSE_W_per_kg':'rmse_W_per_kg',
                    'MAE_W_per_kg':'mae_W_per_kg','R2':'r2'}.get(name,name) if n==11 else name
                ok=ok and np.isclose(float(reference[saved_field]),float(val),rtol=0,atol=1e-11)
        metric_audit.append({'notebook':n,'group':'|'.join(key),'n_predictions':len(rs),
                             'recomputed_RMSE_W_per_kg':measured['RMSE_W_per_kg'],
                             'reported_RMSE_W_per_kg':(reference.get('rmse_W_per_kg') if n==11 else reference.get('RMSE_W_per_kg')) if reference else '',
                             'match':bool(ok)})
        if not ok:issues.append(f'Notebook {n:02d} {key}: pooled metric mismatch')

# Cross-archive checks: downstream source identity and exact Notebook 04 dependency
ref04sha=next(r['sha256'] for r in inventory if r['notebook']==4)
for n,s in summaries.items():
    if s.get('source_sha256') and s['source_sha256']!=AUTHORITATIVE_SHA256:
        issues.append(f'Notebook {n:02d}: incorrect source hash')
    if n in (5,6,7,8,9,10,11,12,13,14):
        candidate_hashes=[v for k,v in s.items() if isinstance(v,str) and ('zip_sha256' in k or k=='reference_zip_sha256' or k=='reference_notebook_04_zip_sha256' or k=='reference_notebook04_zip_sha256')]
        # Some summaries refer primarily to other upstream ZIPs; verify any field explicitly identifying 04
        for k,v in s.items():
            if isinstance(v,str) and ('04' in k.lower() or 'notebook04' in k.lower()) and 'zip_sha256' in k.lower():
                if v!=ref04sha:issues.append(f'Notebook {n:02d}: Notebook 04 fingerprint mismatch ({k})')

# Requirement RC-4 evidence: four comparisons × five outer folds
rc4_rows=[]
if 9 in archives:
    with zipfile.ZipFile(archives[9]) as z:
        timings=read_zip_csv(z,'outer_model_timings.csv')
    for rep in ('W-ALL9','D-ALL9'):
        for est in ('RIDGE','SVR'):
            rr=[r for r in timings if r['representation']==rep and r['estimator']==est]
            ok=(len(rr)==5 and {int(r['outer_fold']) for r in rr}==set(range(1,6))
                and all(float(r['fit_seconds'])>0 and
                        float(r['single_trial_prediction_mean_seconds'])>0 for r in rr))
            if not ok:issues.append(f'RC-4 timing missing or invalid: {rep} {est}')
            rc4_rows.append({'representation':rep,'estimator':est,'n_outer_folds':len(rr),
                'mean_fit_seconds':float(np.mean([float(r['fit_seconds']) for r in rr])) if rr else None,
                'mean_single_trial_prediction_seconds':
                    float(np.mean([float(r['single_trial_prediction_mean_seconds']) for r in rr])) if rr else None,
                'evidence_complete':ok})
else: issues.append('RC-4: Notebook 09 timing archive is missing')

for r in inventory:
    if r['reported_status'] not in ('PASS','MISSING'):
        issues.append(f'Notebook {r["notebook"]:02d}: archive reports {r["reported_status"]}')
    if r['failed_checks']:
        issues.append(f'Notebook {r["notebook"]:02d}: {r["failed_checks"]} failed internal checks')
    if r['reported_status']=='MISSING':
        issues.append(f'Notebook {r["notebook"]:02d}: archive is missing')
print('Pooled prediction groups independently audited:',len(metric_audit))
print('Metric mismatches:',sum(not x['match'] for x in metric_audit))
print('RC-4 conditions recorded:',len(rc4_rows))
print('Open evidence issues:',len(issues))

## Requirements-evidence mapping


In [ ]:
# 8 — Explicit requirement-to-evidence coverage (no automatic blanket approval)
REQUIREMENTS={
 'IR-1':('B0/B1/B2 and development-selected G', [4]),
 'IR-2':('Matched C versus C+G', [5]),
 'IR-3':('Anatomical, W/D/P and descriptor-family comparisons', [4,7,8]),
 'IR-4':('Separate 285-trial eligibility sensitivity', [12]),
 'IR-5':('Fixed-W/D Ridge versus RBF-SVR', [9]),
 'IR-6':('Matched 8, 12 and full-development training budgets', [10]),
 'IR-7':('Participant/condition diagnostics, balanced RMSE and primary intervals', [13,14]),
 'IR-8':('Fixed-ledger participant influence and selection stability', [14]),
 'IR-9':('K=3 and K=6 personal-calibration comparisons', [11]),
 'IR-10':('C+B versus C+B+G comparison', [6]),
 'EC-1':('Corrected source SHA-256 and exact primary ledger', [2,4]),
 'EC-2':('Finite positive targets and nine finite 101-phase angle matrices', [2]),
 'EC-3':('Training-only transforms and participant-disjoint fitting', [3,4,5,6,7,8,9,10,11,12]),
 'EC-4':('Authorised predictors only for each input contract', [3,4,5,6,7,8,9,10,11,12]),
 'EC-5':('One finite, aligned prediction per evaluation trial', [4,5,6,7,8,9,10,11,12]),
 'RC-1':('Reproducible source, code, configuration and run archives', list(range(2,15))),
 'RC-2':('Independent saved-prediction metric recalculation', [4,5,6,7,8,9,10,11,12,13,14]),
 'RC-3':('Controlled complete primary run repeatability and recorded numerical tolerance', [4,15]),
 'RC-4':('Fit and per-trial prediction timings for Ridge/SVR by W/D', [9])}
require(len(REQUIREMENTS)==19,'Must track 19 distinct Chapter 3 requirements')
registry=[]
for req,(description,deps) in REQUIREMENTS.items():
    missing=[x for x in deps if x!=15 and x not in archives]
    status='EVIDENCE_AVAILABLE_REPORT_PENDING'
    if missing:status='MISSING_EVIDENCE'
    if issues and any(f'Notebook {x:02d}' in issue for issue in issues for x in deps if x!=15):
        status='REVIEW_REQUIRED'
    if req=='RC-3':
        status=('PROPOSAL_AND_CONFIRMATION_REPORT_PENDING' if confirm_status=='CONFIRMATORY_CHECK_PASS'
                else 'REVIEW_REQUIRED')
    if req=='RC-4' and not all(r['evidence_complete'] for r in rc4_rows):status='REVIEW_REQUIRED'
    if req=='RC-2' and any(not r['match'] for r in metric_audit):status='REVIEW_REQUIRED'
    registry.append({'requirement':req,'evidence_purpose':description,
         'source_notebooks':';'.join(f'{i:02d}' for i in deps),
         'evidence_files':';'.join(ARTIFACTS[i] if i!=15 else ARCHIVE.name for i in deps),
         'evidence_status':status,
         'final_compliance_status':'NOT_YET_FORMALLY_ASSESSED',
         'remaining_action':('Record a justified RC-3 tolerance in Chapter 3, '
                 'then assess confirmation evidence in Chapter 9' if req=='RC-3' else
                 'Reconcile and report findings/verification evidence in Chapters 7–9')})
print('Requirements tracked:',len(registry))
for r in registry:print(r['requirement'],r['evidence_status'])

## Export saved evidence


In [ ]:
# 9 — Export all controlled repeatability and requirements evidence
OUT.mkdir(parents=True,exist_ok=True)
repeat_predictions=pilot['predictions']+confirm['predictions']
comparison_predictions=pilot_comp['prediction_diffs']+confirm_comp['prediction_diffs']
selection_comparisons=pilot_comp['selections']+confirm_comp['selections']
score_comparisons=pilot_comp['score_diffs']+confirm_comp['score_diffs']
summary_rows=[]
for run,comp in ((pilot,pilot_comp),(confirm,confirm_comp)):
    summary_rows.append({'run_id':run['run_id'],'elapsed_seconds':run['elapsed_seconds'],
        'n_predictions':len(run['predictions']),'n_selected_models':len(run['selected']),
        'n_candidate_scores':len(run['scores']),
        'max_abs_prediction_difference_W_per_kg':comp['max_prediction_abs_difference_W_per_kg'],
        'max_abs_inner_score_difference_W_per_kg':comp['max_score_abs_difference_W_per_kg'],
        'all_selected_configs_match':comp['all_selected_configs_match']})

files={
  'upstream_archive_inventory.csv':(inventory,['notebook','filename','sha256','n_files','reported_status',
      'verification_rows','failed_checks','source_sha256_match']),
  'complete_repeat_predictions.csv':(repeat_predictions,['run_id','procedure','outer_fold',
      'subject_id','trial_uid','observed_W_per_kg','predicted_W_per_kg']),
  'reference_prediction_differences.csv':(comparison_predictions,['run_id','procedure','trial_uid',
      'outer_fold','reference_prediction_W_per_kg','repeated_prediction_W_per_kg',
      'absolute_difference_W_per_kg']),
  'model_selection_comparisons.csv':(selection_comparisons,['run_id','outer_fold','procedure',
      'matches_reference','reference_representation','repeated_representation',
      'reference_alpha','repeated_alpha','reference_n_components','repeated_n_components']),
  'inner_candidate_score_differences.csv':(score_comparisons,['run_id','outer_fold','representation',
      'alpha','n_components','absolute_difference_W_per_kg']),
  'repeat_run_summaries.csv':(summary_rows,['run_id','elapsed_seconds','n_predictions',
      'n_selected_models','n_candidate_scores','max_abs_prediction_difference_W_per_kg',
      'max_abs_inner_score_difference_W_per_kg','all_selected_configs_match']),
  'independent_pooled_metric_audit.csv':(metric_audit,['notebook','group','n_predictions',
      'recomputed_RMSE_W_per_kg','reported_RMSE_W_per_kg','match']),
  'rc4_timing_summary.csv':(rc4_rows,['representation','estimator','n_outer_folds',
      'mean_fit_seconds','mean_single_trial_prediction_seconds','evidence_complete']),
  'requirements_evidence_register.csv':(registry,['requirement','evidence_purpose',
      'source_notebooks','evidence_files','evidence_status','final_compliance_status',
      'remaining_action']),
  'evidence_issues.csv':([{'issue':x} for x in issues],['issue'])}
for filename,(records,fields) in files.items():write_csv(OUT/filename,records,fields)

notebook_path=Path('15_Bacek_Computational_Repeatability_Requirements_Audit.ipynb')
notebook_hash=None
if notebook_path.is_file():
    notebook_json=json.loads(notebook_path.read_text(encoding='utf-8'))
    src=''.join(''.join(c['source']) for c in notebook_json['cells'] if c['cell_type']=='code')
    notebook_hash=hashlib.sha256(src.encode('utf-8')).hexdigest()
rc3_record={'status':'PROPOSED_TOLERANCE_AND_CONFIRMATORY_EVIDENCE_ONLY',
            'pilot':pilot_record,
            'confirmatory_status':confirm_status,
            'confirmatory_max_difference_to_reference_W_per_kg':
                 confirm_comp['max_prediction_abs_difference_W_per_kg'],
            'confirmatory_max_difference_to_pilot_W_per_kg':pilot_vs_confirm,
            'criterion_locked_before_confirmation':True,
            'formal_requirement_value_recorded_in_chapter_3':False,
            'compliance_not_declared_until_report_criterion_recorded':True,
            'numerical_precision_not_predictive_accuracy':True}
(OUT/'rc3_tolerance_pilot_and_confirmation.json').write_text(json.dumps(rc3_record,indent=2),encoding='utf-8')
all_files_present=len(archives)==13
all_evidence_healthy=(all_files_present and len(issues)==0 and
                      all(x['match'] for x in metric_audit) and
                      all(x['evidence_complete'] for x in rc4_rows))
summary={'status':('EVIDENCE_READY_FOR_REPORT_REVIEW' if
                 confirm_status=='CONFIRMATORY_CHECK_PASS' and all_evidence_healthy
                 else 'REVIEW_REQUIRED'),
         'notebook':'15_Bacek_Computational_Repeatability_Requirements_Audit.ipynb',
         'notebook_code_sha256':notebook_hash,
         'source_sha256':AUTHORITATIVE_SHA256,
         'reference_notebook04_archive_sha256':ref04sha,
         'rc3':rc3_record,'n_complete_repeats':2,
         'trial_population':{'n_trials':284,'n_participants':21,'n_cycles':60757},
         'n_reference_predictions':len(reference_ledger),
         'n_prediction_comparisons':len(comparison_predictions),
         'n_model_selection_comparisons':len(selection_comparisons),
         'n_candidate_score_comparisons':len(score_comparisons),
         'upstream_archives_present':len(archives),
         'upstream_archives_expected':13,
         'independently_recalculated_pooled_groups':len(metric_audit),
         'metric_mismatches':sum(not x['match'] for x in metric_audit),
         'evidence_issue_count':len(issues),
         'n_requirements':len(registry),
         'final_compliance_claimed':False,
         'software':PACKAGE_VERSIONS,
         'threadpool_info':THREADPOOLS,
         'thread_environment':{v:os.environ.get(v) for v in
             ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS')},
         'reference_software_matches':env_matches,
         'started_at_utc':pilot['started_at_utc'],
         'finished_at_utc':datetime.now(timezone.utc).isoformat()}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('\nSaved:',ARCHIVE,'(',len(files)+2,'evidence files )')
print('Status:',summary['status'])
print('RC-3 tolerance candidate:',proposed_tolerance,'W/kg')
print('RC-3 confirmation:',confirm_status,'— formal Chapter 3 criterion still pending')
print('All 19 requirement rows saved; final compliance NOT declared.')